## Install Libraries

Runs the end-to-end stage of the GRF pipeline (architecture §8, §10) on a
candidate file produced by `retrieval.ipynb` or `baseline.ipynb`: load the saved
candidates from Google Drive, apply the configured re-ranker (Top-1 or a
language model), and report end-to-end accuracy.

This notebook does **not** load the encoder, the vector index, or the KB
artifact, and never re-runs retrieval. It needs no GPU unless
`LLM_BACKEND = "local"` and `LLM_LOCAL_DEVICE = "cuda"`.

Sources: `docs/architecture.md`, `paper/reference_abstract.md`,
`source/GRF/README.md`.

`tqdm` drives the progress bars; `transformers` (and Colab's `torch`) back the
local re-ranker and are unused when `RERANK = "top1"`.

In [ ]:
%pip install "tqdm>=4.66" "transformers>=4.44"

## Import Libraries

Third-party imports only. The `GRF` package lives inside the repository clone,
so project modules are imported at the end of the repository-setup stage below.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

## Constants and Configuration

Set `DRIVE_DATA_DIR` once; the paths below derive from it:

- `CANDIDATES_PATH`: the single JSONL file written by `retrieval.ipynb` or
  `baseline.ipynb`,
- `KB_JSONL_PATH`: KB data, used only for preferred names when
  `RERANK = "llm"`,
- `RESULTS_OUTPUT_PATH`: end-to-end metrics written by this notebook.

`RERANK = "top1"` uses `PickTopOne`; `"llm"` uses a language model to pick one
CUI and falls back to Top-1 on any error (architecture §8).

**Re-ranker model.** `LLM_BACKEND = "local"` loads `LLM_MODEL_LOCAL` through
`transformers` (Hub id or a local directory); `"api"` uses the
OpenAI-compatible client and needs `OPENAI_API_KEY`.

In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive inputs and output (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
DRIVE_DATA_DIR = DRIVE_ROOT / "REPLACE_ME"                      # TODO: set once
CANDIDATES_PATH = DRIVE_DATA_DIR / "grf_candidates.jsonl"       # from retrieval.ipynb
KB_JSONL_PATH = DRIVE_DATA_DIR / "kb.jsonl"                     # only for RERANK="llm"
RESULTS_OUTPUT_PATH = DRIVE_DATA_DIR / "grf_e2e_results.json"   # metrics output

# ---------------------------------------------------------------------------
# Re-ranking (architecture §8)
# ---------------------------------------------------------------------------
RERANK = "top1"                 # "top1" or "llm"

# ---------------------------------------------------------------------------
# Re-ranker model: local HuggingFace weights (default) or the OpenAI-compatible API
# ---------------------------------------------------------------------------
LLM_BACKEND = "local"           # "local" (transformers) or "api"

# --- "local": loaded through transformers; HF id or a local directory path ---
LLM_MODEL_LOCAL = "Qwen/Qwen2.5-7B-Instruct"
LLM_LOCAL_DEVICE = "auto"       # "auto", "cuda", "cpu" or "cuda:0"
LLM_LOCAL_DTYPE = "auto"        # "auto", "float16", "bfloat16" or "float32"
LLM_LOCAL_MAX_NEW_TOKENS = 64   # the answer is one CUI, so 64 tokens is plenty
LLM_LOCAL_TEMPERATURE = 0.0     # 0.0 = greedy (deterministic)
LLM_LOCAL_TRUST_REMOTE_CODE = False
LLM_LOCAL_USE_CHAT_TEMPLATE = True   # False = feed the raw prompt

# --- "api": OpenAI-compatible HTTP endpoint; needs OPENAI_API_KEY ---
LLM_MODEL_API = "gpt-4o"
LLM_API_BASE_URL = ""           # empty -> OPENAI_BASE_URL env var / api.openai.com
LLM_API_TIMEOUT_S = 30.0
# Set the key in this runtime (never commit it):
#   import getpass, os; os.environ["OPENAI_API_KEY"] = getpass.getpass()

LLM_ACTIVE_MODEL = LLM_MODEL_LOCAL if LLM_BACKEND == "local" else LLM_MODEL_API

# ---------------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------------
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"Candidates   : {CANDIDATES_PATH}")
print(f"KB data      : {KB_JSONL_PATH} (used only when RERANK='llm')")
print(f"Results out  : {RESULTS_OUTPUT_PATH}")
print(f"Re-rank      : {RERANK}")
if RERANK == "llm":
    print(f"Re-rank LLM  : backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL}")

## Stage 1 — Clone or Update the GRF Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the project modules from `source/GRF/`. Re-running this cell
updates the existing clone instead of downloading it again.

In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "GRF" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'GRF' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from GRF.candidates import load_candidates, rerank_candidates
from GRF.evaluator import metrics_from_responses
from GRF.hf_llm import TransformersChatClient
from GRF.kb import load_concepts_jsonl
from GRF.llm import OpenAICompatibleClient
from GRF.rerank import LLMReranker, PickTopOne

log(f"Repository ready at {REPO_CLONE_DIR}; GRF imported from {SOURCE_DIR}.")


def build_llm_client(backend, model):
    """Local transformers model or the OpenAI-compatible API client."""
    if backend == "local":
        return TransformersChatClient(
            model,
            device=LLM_LOCAL_DEVICE,
            dtype=LLM_LOCAL_DTYPE,
            max_new_tokens=LLM_LOCAL_MAX_NEW_TOKENS,
            temperature=LLM_LOCAL_TEMPERATURE,
            trust_remote_code=LLM_LOCAL_TRUST_REMOTE_CODE,
            use_chat_template=LLM_LOCAL_USE_CHAT_TEMPLATE,
        )
    if backend == "api":
        if not os.environ.get("OPENAI_API_KEY"):
            log(
                "WARNING: OPENAI_API_KEY is not set. API calls will fail and "
                "every mention will fall back to Top-1."
            )
        return OpenAICompatibleClient(
            model, base_url=LLM_API_BASE_URL or None, timeout_s=LLM_API_TIMEOUT_S
        )
    raise ValueError(f"LLM_BACKEND must be 'local' or 'api', got {backend!r}")


log("LLM factory ready; re-rank backend=%s." % LLM_BACKEND)

## Stage 2 — Mount Google Drive and Locate Inputs

The candidate file lives on Drive. This stage mounts Drive, checks it exists,
and validates the output path. The KB JSONL is required only for
`RERANK = "llm"`.

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

candidates_path = Path(require_configured(CANDIDATES_PATH, "CANDIDATES_PATH"))
results_path = Path(require_configured(RESULTS_OUTPUT_PATH, "RESULTS_OUTPUT_PATH"))
if not candidates_path.is_file():
    raise FileNotFoundError(f"Candidate file not found: {candidates_path}")
log(
    f"Candidates: {candidates_path} "
    f"({candidates_path.stat().st_size / 1024 ** 2:.2f} MB)"
)

if RERANK == "llm":
    kb_path = Path(require_configured(KB_JSONL_PATH, "KB_JSONL_PATH"))
    if not kb_path.is_file():
        raise FileNotFoundError(f"KB JSONL not found: {kb_path}")
    log(f"KB data: {kb_path} ({kb_path.stat().st_size / 1024 ** 2:.2f} MB)")
else:
    kb_path = None
    log("RERANK='top1': KB data not needed.")

if results_path.suffix.lower() != ".json":
    raise ValueError(
        f"RESULTS_OUTPUT_PATH must end in .json, got {results_path.name!r}."
    )
log(f"Results output: {results_path}")

## Stage 3 — Load Saved Candidates

`load_candidates` restores the gold samples and the retrieval responses from the
single JSONL file. Nothing is re-encoded or re-retrieved. It reports each
record through `progress_callback`, which drives the bar.

In [ ]:
log(f"Loading candidates from {candidates_path} ...")
started = time.time()
with progress_bar(None, "Loading candidate records", "record") as report:
    samples, responses = load_candidates(candidates_path, progress_callback=report)
log(
    f"Loaded {len(samples):,} samples and {len(responses):,} responses "
    f"in {time.time() - started:.1f}s."
)
if not samples:
    raise ValueError(f"{candidates_path} contains no candidate records.")
empty = sum(1 for response in responses if not response.candidates)
log(f"Mentions with candidates: {len(responses) - empty:,}; without: {empty:,}.")

## Stage 4 — Load Concept Names (LLM Re-ranker Only)

The LLM re-ranker sees preferred names from the KB data. With
`RERANK = "top1"`, names are not used and this stage is skipped.

In [ ]:
if RERANK == "llm":
    if kb_path is None:
        raise ValueError("RERANK='llm' requires KB_JSONL_PATH.")
    log(f"Loading KB concept names from {kb_path} ...")
    started = time.time()
    with progress_bar(None, "Parsing KB records", "record") as report:
        kb_concepts = load_concepts_jsonl(kb_path, progress_callback=report)
    if not kb_concepts:
        raise ValueError(f"{kb_path} produced no KB concepts.")
    concept_names = {concept.cui: concept.name for concept in kb_concepts}
    log(
        f"Loaded {len(concept_names):,} preferred names "
        f"in {time.time() - started:.1f}s."
    )
else:
    concept_names = {}
    log("RERANK='top1': concept names are not needed.")

## Stage 5 — Build the Re-ranker

`PickTopOne` keeps the highest-scoring candidate (CUI-ascending tie-break).
`LLMReranker` asks the configured model to pick one CUI; a failed or ambiguous
answer falls back to Top-1 (architecture §6, §8).

The re-ranker client follows `LLM_BACKEND`: a local `transformers` model
(`LLM_MODEL_LOCAL`) or the OpenAI-compatible API. The local client has no
`timeout_s` (a local generation cannot be interrupted safely), so the timeout
argument is forwarded only to the API client.

In [ ]:
if RERANK == "llm":
    log(
        f"Re-rank LLM: backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL!r} "
        f"(weights load lazily on the first call)."
    )
    llm = build_llm_client(LLM_BACKEND, LLM_ACTIVE_MODEL)
    reranker = LLMReranker(
        llm, timeout_s=None if LLM_BACKEND == "local" else LLM_API_TIMEOUT_S
    )
else:
    llm = None
    reranker = PickTopOne()

log(f"Re-ranker ready: {type(reranker).__name__}.")

## Stage 6 — Run End-to-End Re-ranking

`rerank_candidates` applies the re-ranker to each saved candidate list and
reports each completed mention through `progress_callback`; the bar reflects
real completed work.

In [ ]:
with progress_bar(len(samples), "Re-ranking candidates", "mention") as report:
    started = time.time()
    reranked = rerank_candidates(
        samples,
        responses,
        reranker=reranker,
        concept_names=concept_names,
        progress_callback=report,
    )

log(f"Re-ranked {len(reranked):,} mentions in {time.time() - started:.1f}s.")
failures = [
    response.debug["rerank"]["error"]
    for response in reranked
    if isinstance(response.debug, dict)
    and (response.debug.get("rerank") or {}).get("error")
]
if failures:
    log(f"WARNING: {len(failures):,} re-rank failure(s); first: {failures[0]}")

## Stage 7 — End-to-End Metrics

Accuracy is the fraction of mentions whose final re-ranked CUI equals the gold
CUI (architecture §10). R@1/R@5 are shown for reference and come from the saved
retrieval ranking, so they match `retrieval.ipynb`.

In [ ]:
overall, by_label = metrics_from_responses(samples, reranked)


def _row(label, metrics) -> str:
    return (
        f"{str(label):<16}{metrics.n:>7}"
        f"{100 * metrics.recall_at_1:>10.2f}{100 * metrics.recall_at_5:>10.2f}"
        f"{100 * metrics.accuracy:>10.2f}"
    )


print(f"{'label':<16}{'n':>7}{'R@1 %':>10}{'R@5 %':>10}{'acc %':>10}")
print(_row("ALL", overall))
for label, metrics in by_label.items():
    print(_row(label, metrics))

## Stage 8 — Save End-to-End Results

Write the candidate-file provenance, re-ranker configuration, overall metrics
and per-label metrics to the single configured `RESULTS_OUTPUT_PATH`, then read
it back to confirm the file is usable.

In [ ]:
payload = {
    "candidates_file": str(candidates_path),
    "n_samples": len(samples),
    "rerank": RERANK,
    "llm_backend": LLM_BACKEND if RERANK == "llm" else None,
    "llm_model": LLM_ACTIVE_MODEL if RERANK == "llm" else None,
    "rerank_failures": len(failures),
    "overall": asdict(overall),
    "by_label": {str(label): asdict(metrics) for label, metrics in by_label.items()},
}

results_path.parent.mkdir(parents=True, exist_ok=True)
log(f"Saving results to {results_path} ...")
results_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
saved = json.loads(results_path.read_text(encoding="utf-8"))
assert saved["overall"] == payload["overall"]
log(
    f"Saved and verified {results_path} "
    f"({results_path.stat().st_size / 1024 ** 2:.2f} MB)."
)
log("End-to-end evaluation complete.")

## Stage 9 — Release Memory

The results file is written and verified, so unload the re-ranker's local
weights (if any), drop the candidate lists and the KB names, then empty the
CUDA cache.

In [ ]:
if "llm" in globals() and llm is not None and hasattr(llm, "unload"):
    llm.unload()
    log("Local re-ranker weights unloaded.")

for name in (
    "samples",
    "responses",
    "reranked",
    "concept_names",
    "kb_concepts",
    "reranker",
    "llm",
    "failures",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass

log("E2E memory released (samples, responses, re-ranked responses).")